In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from pathlib import Path
import polars as pl

event_frames = []
MATCH_COUNT = 100
counter = 0

for file in Path('data/events').glob('*.json'):
    counter += 1
    if counter > MATCH_COUNT:
        break
    event_frames.append(
        pl.read_json(file, infer_schema_length=None).with_columns(
            pl.lit(int(file.stem), dtype=pl.Int64).alias('match_id')
        )
    )

events_df = pl.concat(event_frames, how='diagonal_relaxed')
base_columns = [
    'id', 'index', 'period', 'timestamp', 'minute', 'second', 'type', 'possession',
    'possession_team', 'play_pattern', 'team', 'duration', 'tactics', 'match_id',
    'related_events', 'player', 'position', 'location',
]
events_df = events_df.select(base_columns + [
    column for column in events_df.columns if column not in base_columns
])

In [10]:
match_frames = []
# load matches df from folders in matches folder
subfolders = [f for f in Path('data/matches/').iterdir() if f.is_dir()]
for subfolder in subfolders:
    for file in subfolder.glob('*.json'):
        match_frames.append(pl.read_json(file, infer_schema_length=None).with_columns(
            pl.lit(int(file.stem), dtype=pl.Int64).alias('season_id')
        ))

matches_df = pl.concat(match_frames, how='diagonal_relaxed')

matches

In [11]:
all_teams = events_df.select('team').unique().to_series().to_list()
all_players = events_df.select('player').unique().to_series().to_list()
all_competitions = matches_df.select('competition').unique().to_series().to_list()
all_seasons = matches_df.select('season').unique().to_series().to_list()

In [12]:
teams_dict = {team['id']: team['name'] for team in all_teams}
players_dict = {player['id']: player['name'] for player in all_players if isinstance(player, dict)}
competitions_dict = {competition['competition_id']: competition['competition_name'] for competition in all_competitions if isinstance(competition, dict)}
competitions_country_dict = {competition['competition_id']: competition['country_name'] for competition in all_competitions if isinstance(competition, dict)}
seasons_dict = {season['season_id']: season['season_name'] for season in all_seasons if isinstance(season, dict)}

In [13]:
# replace competition column with its id
matches_df = matches_df.with_columns(
    pl.col('competition').map_elements(lambda x: x['competition_id'] if isinstance(x, dict) else None).alias('competition_id')
).drop('competition')

In [14]:
premier_league_matches_df = matches_df.filter(pl.col('competition_id') == 2)
bundesliga_matches_df = matches_df.filter(pl.col('competition_id') == 9)    

In [ ]:
id_season_15_16 = 27
premier_league_season_15_16_matches_df = premier_league_matches_df.filter(pl.col('season_id') == id_season_15_16)
premier_league_season_15_16_matches = premier_league_season_15_16_matches_df.select('match_id').to_series().to_list()
# write premier_league_season_15_16_matches ids to csv
premier_league_season_15_16_matches_df.select(['match_id']).unique().write_csv('premier_league_season_15_16_matches.csv')

In [ ]:
# write competitions, seasons, to csv
competitions_df = pl.DataFrame({
    'competition_id': list(competitions_dict.keys()),
    'competition_name': list(competitions_dict.values()),
    'country_name': [competitions_country_dict.get(k) for k in competitions_dict.keys()]
})
competitions_df.write_csv('competitions.csv')

seasons_df = pl.DataFrame({
    'season_id': list(seasons_dict.keys()),
    'season_name': list(seasons_dict.values())
})
seasons_df.write_csv('seasons.csv')



In [ ]:
non_playing_events_ids = [5, 18, 19, 26, 27, 29, 34, 35, 36, 40]
# rename carry to carry_to
events_df = events_df.rename({'carry': 'carry_to'})
def set_to_id(value):
    return int(value['id']) if isinstance(value, dict) and 'id' in value else value

id_cols = ['type', 'possession_team', 'play_pattern', 'team', 'player', 'position']
events_df = events_df.with_columns(
    pl.col(column).map_elements(set_to_id, return_dtype=pl.Int64).alias(column)
    for column in id_cols
)

In [ ]:
cols_to_expand = events_df.columns[18:]
for col in cols_to_expand:
    dtype = events_df.schema[col]
    if isinstance(dtype, pl.Struct):
        events_df = events_df.with_columns(
            pl.col(col).struct.field(field.name).alias(f'{col}_{field.name}')
            for field in dtype.fields
        ).drop(col)

for col in events_df.columns:
    dtype = events_df.schema[col]
    if isinstance(dtype, pl.Struct) and 'id' in {field.name for field in dtype.fields}:
        events_df = events_df.with_columns(pl.col(col).struct.field('id').alias(col))


In [ ]:
playing_events_df = events_df.filter(~pl.col('type').is_in(non_playing_events_ids))
playing_events_df = playing_events_df.drop('tactics')
nAn = playing_events_df.get_column('block_deflection').unique().to_list()[0]
print(playing_events_df.schema)

In [ ]:
# print list of columns zero-indexed
for i, col in enumerate(playing_events_df.columns):
    print(f"{i}: {col}")

In [ ]:
from tools_file import *
# load json files 
event_types_dict = json_file_to_dict('info/event_type.json')
positions_dict = json_file_to_dict('info/position.json')
positions_abbrev_dict = json_file_to_dict('info/position.json', value_field='abbrev')
play_patterns_dict = json_file_to_dict('info/play_pattern.json')

In [ ]:
event_types_dict[33] = '50-50'

In [ ]:
end_loc_cols = [s for s in playing_events_df.columns if "end_location" in s]
outcome_cols = [s for s in playing_events_df.columns if "outcome" in s]
type_cols = [s for s in playing_events_df.columns if "_type" in s]
technique_cols = [s for s in playing_events_df.columns if "technique" in s]
body_part_cols = [s for s in playing_events_df.columns if "body_part" in s]
aerial_won_cols = [s for s in playing_events_df.columns if "aerial_won" in s]

playing_events_df = playing_events_df.with_columns(
    pl.coalesce(end_loc_cols).alias('end_location'),
    pl.coalesce(outcome_cols).alias('outcome'),
    pl.coalesce(type_cols).alias('action_type'),
    pl.coalesce(technique_cols).alias('technique'),
    pl.coalesce(body_part_cols).alias('body_part'),
    pl.coalesce(aerial_won_cols).alias('aerial_won'),
).drop(end_loc_cols + outcome_cols + type_cols + technique_cols + body_part_cols + aerial_won_cols)

all_null_columns = [
    column for column in playing_events_df.columns
    if playing_events_df.get_column(column).null_count() == playing_events_df.height
]
playing_events_df = playing_events_df.drop(all_null_columns)

playing_events_df = playing_events_df.with_columns(
    (pl.lit(72) - pl.col('foul_committed_card')).alias('foul_committed_card'),
    (pl.lit(72) - pl.col('bad_behaviour_card')).alias('bad_behaviour_card'),
)

# collapse body part cols
clearance_cols = [s for s in playing_events_df.columns if "clearance_" in s]

playing_events_df = playing_events_df.drop(clearance_cols)


In [ ]:
import numpy as np

def split_location(df, col):
    return df.with_columns(
        pl.col(col).list.get(0, null_on_oob=True).alias(f'{col}_x'),
        pl.col(col).list.get(1, null_on_oob=True).alias(f'{col}_y'),
    )

playing_events_df = split_location(playing_events_df, 'location')
playing_events_df = split_location(playing_events_df, 'end_location')
playing_events_df = playing_events_df.drop(['location', 'end_location'])

# get time col and take exact millisecond of event
playing_events_df = playing_events_df.with_columns(
    pl.col('timestamp').str.extract(r'\.(\d+)', 1).fill_null('0').cast(pl.Int64).alias('millisecond')
)


In [ ]:
import matplotlib.pyplot as plt
# plot event type distribution
type_counts = playing_events_df.group_by('type').len().sort('len', descending=True)
type_counts = type_counts.with_columns(
    pl.col('type').replace_strict(event_types_dict, default=None).alias('name')
)
plt.bar(type_counts['name'].to_list(), type_counts['len'].to_list())
plt.title("Event Type Distribution")
plt.xlabel("Event Type")
plt.xticks(rotation=90, ha='right')
plt.ylabel("Count")


In [ ]:
# look at one match
playing_events_df_42 = playing_events_df.filter(pl.col('match_id') == 42)

In [ ]:
attack_events_ids = [14, 16, 30, 43] # dribble / shot / pass / carry
defend_events_ids = [2, 6, 9, 10, 17, 22] # ball recovery / block / clearance / interception / press / foul committed
combat_events_ids = [4, 28, 33] # duel / shield / 50/50
errors_events_ids = [3, 8, 37, 38, 39] # dispossessed / offside / error / miscontrol / dribbled past
ball_receipts_ids = [42]

In [ ]:
# split the pitch into zones and assign each event to a zone based on its location
num_x_zones = 20
num_y_zones = 8

x_bins = np.linspace(0, 120, num_x_zones + 1)[:-1]
y_bins = np.linspace(0, 80, num_y_zones + 1)[:-1]

xy_bins = np.array(np.meshgrid(x_bins, y_bins)).T.reshape(-1, 2)
attack_xy_bins_dict = {(x, y): {z: 0 for z in attack_events_ids} for x, y in xy_bins}
defend_xy_bins_dict = {(x, y): {z: 0 for z in defend_events_ids} for x, y in xy_bins}
combat_xy_bins_dict = {(x, y): {z: 0 for z in combat_events_ids} for x, y in xy_bins}
errors_xy_bins_dict = {(x, y): {z: 0 for z in errors_events_ids} for x, y in xy_bins}
ball_receipts_xy_bins_dict = {(x, y): {42: 0} for x, y in xy_bins}

In [ ]:
def fit_to_zone(x, y):
    if x is None or y is None:
        return None
    return (int(np.digitize(x, x_bins) - 1), int(np.digitize(y, y_bins) - 1))

playing_events_df = playing_events_df.with_columns(
    pl.struct(["location_x", "location_y"]).map_elements(
        lambda s: list(fit_to_zone(s["location_x"], s["location_y"])) if fit_to_zone(s["location_x"], s["location_y"]) is not None else None,
        return_dtype=pl.List(pl.Int64),
    ).alias("zone")
).with_columns(
    pl.col("zone").list.get(0).alias("x_zone"),
    pl.col("zone").list.get(1).alias("y_zone"),
).drop("zone")

In [ ]:
# fill bin dictionaries with counts of events in each zone
for x in range(num_x_zones):
    x_id = x_bins[x]
    for y in range(num_y_zones):
        y_id = y_bins[y]
        zone_df = playing_events_df.filter(
            (pl.col("x_zone") == x) & (pl.col("y_zone") == y)
        )
        for event_id in attack_events_ids:
            attack_xy_bins_dict[(x_id, y_id)][event_id] = zone_df.filter(pl.col("type") == event_id).height
        for event_id in defend_events_ids:
            defend_xy_bins_dict[(x_id, y_id)][event_id] = zone_df.filter(pl.col("type") == event_id).height
        for event_id in combat_events_ids:
            combat_xy_bins_dict[(x_id, y_id)][event_id] = zone_df.filter(pl.col("type") == event_id).height
        for event_id in errors_events_ids:
            errors_xy_bins_dict[(x_id, y_id)][event_id] = zone_df.filter(pl.col("type") == event_id).height
        for event_id in ball_receipts_ids:
            ball_receipts_xy_bins_dict[(x_id, y_id)][event_id] = zone_df.filter(pl.col("type") == event_id).height

In [ ]:
# plot pitch zones with event counts for attack events
from mplsoccer import Pitch
from matplotlib import pyplot as plt

def plot_event_counts(event_dict, event_id):
    pitch = Pitch(pitch_type='statsbomb', axis=True, label=True)
    fig, ax = pitch.draw()
    specific_id = event_id # e.g. dribble

    shapes = []
    counts = []
    # plot blocks on pitch from dict counts
    for x in range(num_x_zones):
        x_id = x_bins[x]
        for y in range(num_y_zones):
            y_id = y_bins[y]
            # generate shape for each zone based on its x and y coordinates
            shapes.append(np.array([[x_id, y_id], [x_id + 120/num_x_zones, y_id], [x_id + 120/num_x_zones, y_id + 80/num_y_zones], [x_id, y_id + 80/num_y_zones]]))
            count = event_dict[(x_id, y_id)][specific_id]
            counts.append(count)

    max_count = max(counts)

    for i, shape in enumerate(shapes):
        count = counts[i]
        if count > 0:
            pitch.polygon([shape], color='red', alpha=count / max_count, ax=ax, ec='none', lw=1)

    plt.title(f"Event Counts for {event_types_dict[specific_id]} (id{specific_id}) by Pitch Zone")

plot_event_counts(defend_xy_bins_dict, 10) # interception

In [ ]:
# save images for every event type in each category
import os
stem = f"outputs/zones_{MATCH_COUNT}"
os.makedirs(stem, exist_ok=True)

for event_id in attack_events_ids:
    plot_event_counts(attack_xy_bins_dict, event_id)
    plt.gca()
    plt.savefig(f"{stem}/attack_event_{event_id}_{event_types_dict[event_id]}.png")
    plt.close()

for event_id in defend_events_ids:
    plot_event_counts(defend_xy_bins_dict, event_id)
    plt.savefig(f"{stem}/defend_event_{event_id}_{event_types_dict[event_id]}.png")
    plt.close()

for event_id in combat_events_ids:
    plot_event_counts(combat_xy_bins_dict, event_id)
    plt.savefig(f"{stem}/combat_event_{event_id}_{event_types_dict[event_id]}.png")
    plt.close()

for event_id in errors_events_ids:
    plot_event_counts(errors_xy_bins_dict, event_id)
    plt.savefig(f"{stem}/errors_event_{event_id}_{event_types_dict[event_id]}.png")
    plt.close()

for event_id in ball_receipts_ids:
    plot_event_counts(ball_receipts_xy_bins_dict, event_id)
    plt.savefig(f"{stem}/ball_event_42_Receipt.png")
    plt.close()